# 🗣️ Kokoro Yorùbá TTS v1 — Interactive Gradio Demo

Click **Runtime → Run all** (or run the two cells below) to launch a live **Gradio** web UI where you can type any Yorùbá sentence and hear it in 24 kHz audio.

- **Model:** [Akintunde102/kokoro-yoruba-v1](https://huggingface.co/Akintunde102/kokoro-yoruba-v1)
- **G2P Engine:** [`oro-g2p`](https://pypi.org/project/oro-g2p/)
- *Tip: Include Yorùbá diacritics and tone marks (`ẹ, ọ, ṣ, á, à`) for accurate tones.*

In [ ]:
import os, subprocess

if not os.path.exists("/content/StyleTTS2"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/yl4579/StyleTTS2.git", "/content/StyleTTS2"], check=True)

subprocess.run([
    "pip", "install", "--quiet",
    "oro-g2p==0.1.2", "munch", "torchdiffeq", "einops", "einops-exts",
    "huggingface_hub", "gradio>=4.44.0", "soundfile", "pyyaml",
], check=True)
print("✅ Dependencies and StyleTTS2 ready!")

In [ ]:
import sys, types
import numpy as np
import gradio as gr
from huggingface_hub import snapshot_download
from oro_g2p import g2p

# Inference does not need monotonic_align Cython compilation
try:
    import monotonic_align  # noqa: F401
except ImportError:
    ma = types.ModuleType("monotonic_align")
    ma.maximum_path = lambda *a, **k: None
    ma.mask_from_lens = lambda *a, **k: None
    ma_core = types.ModuleType("monotonic_align.core")
    ma_core.maximum_path_c = lambda *a, **k: None
    sys.modules["monotonic_align"] = ma
    sys.modules["monotonic_align.core"] = ma_core

release_dir = snapshot_download(repo_id="Akintunde102/kokoro-yoruba-v1")
if release_dir not in sys.path:
    sys.path.insert(0, release_dir)

from tts_yoruba import YorubaTTS
tts = YorubaTTS(release_dir=release_dir, styletts2_dir="/content/StyleTTS2")

VOICES = {
    "Spk 8 · Studio (BibleTTS)": "spk8",
    "Spk 2 · Conversational (WaxalNLP)": "spk2",
}

PRESETS = [
    ("👋 Greeting", "Ẹ kú àárọ̀ o! Báwo ni àwọn ará ilé? Àlàáfíà ni gbogbo wa wà."),
    ("⚡ Urgent", "Jọ̀wọ́, wá síbí kíákíá!"),
    ("🎯 Tone Triplets", "Ọkọ̀ mi wà ní ọjà, ṣùgbọ́n ọkọ mi fẹ́ ra ọ̀kọ̀ tuntun."),
    ("🎵 Nasals", "Ọ̀rọ̀ Yorùbá dùn láti gbọ́, nítorí pé ohùn rẹ̀ ní ìwọ̀ntúnwọ̀nsì àti ẹwà."),
    ("❓ Question", "Níbo ni o ti ń bọ̀ láti òwúrọ̀ yìí, ṣé o sì ti jẹun?"),
    ("📜 Proverb", "Àgbàlagbà kì í wà lọ́jà, kí orí ọmọ tuntun wọ́."),
    ("📰 News", "Àjọ tó ń bójú tó ètò ẹ̀kọ́ kéde pé gbogbo ilé-ìwé yóò ṣí ní ọjọ́ Ajé tó ń bọ̀."),
    ("📖 Long Story", "Láti ìlú Ìbàdàn títí dé Èkó, àwọn oníṣòwò ń lo ẹ̀rọ ìbánisọ̀rọ̀ láti ta ọjà wọn lórí ayélujára, èyí tí ó sì ń ràn wọ́n lọ́wọ́ láti mú owó wọlé pẹ̀lú ìrọ̀rùn àti àlàáfíà."),
    ("💼 Loanwords", "Mo wà ní míítìǹì ní ọ́fíìsì báyìí, ṣùgbọ́n màá ṣẹ́ẹ̀kì fóònù mi nígbà tí mo bá parí."),
]

def synthesize(text, voice_label, speed):
    text = (text or "").strip()
    if not text:
        return None, ""
    voice_id = VOICES.get(voice_label, "spk8")
    ipa = g2p(text)
    wav = tts.speak(text, voice=voice_id, speed=float(speed))
    wav = np.clip(wav, -1.0, 1.0).astype(np.float32)
    return (24000, wav), ipa

with gr.Blocks(title="Kokoro Yorùbá TTS v1") as demo:
    gr.Markdown(
        "# 🗣️ Kokoro Yorùbá TTS (v1 Live Demo)\n"
        "Click any preset chip below (no scrolling!) or type your own tone-marked Yorùbá text, then click **Generate Speech**."
    )

    preset_btns = []
    with gr.Row():
        for label, sample_txt in PRESETS[:5]:
            preset_btns.append((gr.Button(label, size="sm", min_width=105), sample_txt))
    with gr.Row():
        for label, sample_txt in PRESETS[5:]:
            preset_btns.append((gr.Button(label, size="sm", min_width=105), sample_txt))

    with gr.Row():
        with gr.Column(scale=1, min_width=320):
            text_in = gr.Textbox(label="Yorùbá Text (with tone marks)", lines=4, value=PRESETS[0][1])
            voice_in = gr.Radio(choices=list(VOICES.keys()), value=list(VOICES.keys())[0], label="Voice")
            speed_in = gr.Slider(minimum=0.75, maximum=1.25, value=1.0, step=0.05, label="Speed")
            run_btn = gr.Button("Generate Speech", variant="primary")
        with gr.Column(scale=1, min_width=320):
            audio_out = gr.Audio(label="Generated Yorùbá Audio (24 kHz)", type="numpy")
            ipa_out = gr.Textbox(label="oro-g2p Phonemes (IPA + Tones)", lines=4, interactive=False)

    for btn, sample_txt in preset_btns:
        btn.click(fn=lambda t=sample_txt: t, outputs=text_in)
    run_btn.click(fn=synthesize, inputs=[text_in, voice_in, speed_in], outputs=[audio_out, ipa_out])

demo.queue().launch(share=True, inline=True)